# Test code

### Import of librabries

Check the Gurobi license

In [8]:
import gurobipy as gp
print(gp.gurobi.version())

(12, 0, 3)


In [20]:
import json
import os
from gurobipy import GRB
import networkx as nx
import time as time_setting

### MIP model

Computation of the split coefficients

In [ ]:
def buildNxGraph(net, V):
    G = nx.DiGraph()
    G.add_nodes_from(V)
    for link in net["links"]:
        G.add_edge(link["from"], link["to"], weight=link["metric"])
    return G

def computeSplitCoef(pairs, A, T, V, q_arc, net):
    G = buildNxGraph(net, V)
    r = {}
    
    for time in T:
        G_t = G.copy()
        G_t.remove_edges_from(q_arc.get(time, []))
        
        # Compute dist*(i,j) for every pairs of nodes(i,j)
        shortest_dist = dict(nx.all_pairs_dijkstra_path_length(G_t, weight="weight"))
        
        # Compute dist*(j,i) for every pairs of nodes (i,j)
        shortest_dist_reverse = dict(nx.all_pairs_dijkstra_path_length(G_t.reverse(), weight="weight"))
        
        for (i,j) in pairs:
            if j not in shortest_dist.get(i, {}):
                continue
            
            #d = shortest_dist[i][j]
            shortest_dist_i = shortest_dist[i] 
            FGList = []
            
            # If the arc (u,v) belongs to any shortest path from i to j so it belongs to FG(i,j) i.e.
            # if dist*(i,j) = dist*(i,u) + weight(u,v) + dist*(v,j)
            for (u,v) in G_t.edges():
                if i in shortest_dist and u in shortest_dist[i] and v in shortest_dist_reverse and j in shortest_dist[v]:
                    FGList.append((u,v))
            
            if not FGList:
                continue
            
            FG = nx.DiGraph()
            FG.add_nodes_from(V)
            FG.add_edges_from(FGList)
            
            # ECMP
            # Initialization of the flow
            flow = {node: 0.0 for node in V}
            flow[i] = 1.0
            order = []
            # Store the nodes u which can reach i in oder to store them in a increasing order to compute ECMP
            for u in FG.nodes():
                if u in shortest_dist:
                    order.append((shortest_dist_i[u], u))
            order.sort()
            order = [u for _, u in order]
            
            # Computing of the split coefficients
            for u in order:
                if u == j:
                    continue
                # List of the outgoing edges from u
                out = list(FG.out_edges(u))
                if not out:
                    continue
                # Divide the flow in u by the number of outgoing nodes from u
                split = flow[u] / len(out)
                for _, v in out:
                    # Add the split coefficient in the outgoing nodes and edges from u
                    r[(i, j, u, v, time)] = r.get((i, j, u, v, time), 0.0) + split
                    flow[v] += split
    return r

Creation of the json output

In [58]:
"""
def buildPath(segment_path, sd, td):
    
    # Dictionnary from i to j
    next_node = {i: j for (i, j) in segment_path}
    current = sd
    waypoints = []
    
    while current != td:
        next = next_node.get(current)
        if next is None:
            print("output_creation -> buildPath : Error nxt is None")
            break
        if next != td:
            waypoints.append(next)
        current = next
    return waypoints

"""

# Attention with cycle detection, maybe further in the project, we don't need that
def buildPath(segment_path, sd, td):

    # Dictionnary from i to j
    next_node = {i: j for (i, j) in segment_path}
    current = sd
    waypoints = []
    # Record visisted nodes for cycles
    visited = set()
    
    while current != td:
        
        if current in visited:
            print(f"output_creation -> buildPath : Cycle detected at node {current}")
            break
        visited.add(current)
        
        next = next_node.get(current)
        if next is None:
            print("output_creation -> buildPath : Error next is None")
            break
        if next != td:
            waypoints.append(next)
        current = next
    return waypoints

def output_creation(x, D, T, pairs, s, t):
    srpaths = []
    for d in D:
        sd = s[d]
        td = t[d]
        for time in T:
            segment_path = [(i, j) for (i, j) in pairs if x[d, time, i, j].X > 0.5]

            if not segment_path:
                continue

            waypoints = buildPath(segment_path, sd, td)

            if waypoints:
                srpaths.append({"d": d, "t": time, "w": waypoints})
    return srpaths

Test on instance 1

In [ ]:
idx = 1
datasetPath = "../setA"
t0 = time_setting.time()
if idx < 10:
    instance = f"0{idx}"
else:
    instance = f"{idx}"
outputPath = f"../outputSetA//setA-{instance}-srpaths.json"
print("---------------------------------------------------------------------------------------------------------------------------")
print(f"instance {instance}")
print("---------------------------------------------------------------------------------------------------------------------------")
with open(f"{datasetPath}/setA-{instance}-net.json", "r") as netFile:
    net = json.load(netFile)

with open(f"{datasetPath}/setA-{instance}-scenario.json", "r") as scenarioFile:
    scenario = json.load(scenarioFile)

with open(f"{datasetPath}/setA-{instance}-tm.json", "r") as tmFile:
    tm = json.load(tmFile)

# Network file

V = [node["id"] for node in net["nodes"]]
A = [(link["from"], link["to"]) for link in net["links"]]

# Dict from arc (i, j) to capacity of arc (i, j)
c = {(link["from"], link["to"]): link["capacity"] for link in net["links"]}

# Dict from arc (i, j) to weight/metric of arc (i, j)
weight = {(link["from"], link["to"]): link["metric"] for link in net["links"]}

# Dict from link id to arc (i, j)
link_to_A = {link["id"]: (link["from"], link["to"]) for link in net["links"]}

# Dict from arc (i, j) to link id
A_to_link = {(link["from"], link["to"]): link["id"] for link in net["links"]}


# Scenario file

maxSeg = scenario["max_segments"]

# Dict from time t to budget value at time t
kappla = {i["t"]: i["value"] for i in scenario["budget"]}

# Dict from time t to list of link id (that needs an intervation at time t)
q = {i["t"]: i["links"] for i in scenario["interventions"]}

# Intervation expressed in a arc way instead of the id_link
# Dict from time t to list of arcs (i, j) (that needs an intervation at time t)
q_arc = {t: [link_to_A[link_id] for link_id in links] for t, links in q.items()}

# tm file

num_time_slots = tm["num_time_slots"]
T = list(range(num_time_slots))
T_star = T[1:]
D = list(range(len(tm["demands"])))  # demands indexed 0..n-1
print(f"D : {D}")

# Dict from demand d to source node s(d)
s = {d: tm["demands"][d]["s"] for d in D}
print(f"s: {s}")

# Dict from demand d to target node t(d)
t = {d: tm["demands"][d]["t"] for d in D}
print(f"t: {t}")

# Dict from (d, t) to traffic volume of demand d at time time
v = {(d, time): tm["demands"][d]["v"][time] for d in D for time in T}

# Offline computation

pairs = [(i, j) for i in V for j in V if i != j]

r = computeSplitCoef(pairs, A, T, V, q_arc, net)

print(f"Offline computation: {time_setting.time()-t0} s")

t1 = time_setting.time()

m = gp.Model()

m.Params.OutputFlag = 0 

# Variable definition

# Binary variable: 1 if the path for demand d uses segment (i,j) at time t
x = m.addVars(D, T, V, V, vtype=GRB.BINARY, name="x")

# Load variable of arc a to time t
lambdaVar = m.addVars(A, T, lb=0.0, name="lambda")

# Buffer variable for the absolute value
y = m.addVars(D, T_star, V, V, lb=0.0, name="y")

print(f"Variable definition: {time_setting.time()-t1} s")

t2 = time_setting.time()

# Constraint definition

# (a) Flow conservation constraints
for d in D:
    sd = s[d]
    td = t[d]
    for time in T:
        for i in V:
            buffer = 0
            if i == sd:
                buffer = 1
            elif i == td:
                buffer = -1
            m.addConstr((gp.quicksum(x[d, time, i, j] for j in V if i != j))-gp.quicksum(x[d, time, j, i] for j in V if i != j) == buffer)
print(f"Flow conservation constraints: {time_setting.time()-t2} s")

t3 = time_setting.time()

# (b) Number of segments
m.addConstrs(gp.quicksum(x[d, time, i, j] for (i, j) in pairs) <= maxSeg for d in D for time in T)

print(f"Number of segments constraints: {time_setting.time()-t3} s")

t4 = time_setting.time()

# (c) Load constraints - speeded up - To verify

nonzero_r = {}
for (i, j, a0, a1, time), val in r.items():
    if val > 1e-9:
        key = ((a0, a1), time)
        if key not in nonzero_r:
            nonzero_r[key] = []
        nonzero_r[key].append((i, j, val))

for time in T:
    ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
    for a in ANotInQ:
        terms = nonzero_r.get((a, time), [])
        if not terms:
            continue
        m.addConstr(
            gp.quicksum(
                r_val * v[d, time] * x[d, time, i, j]
                for (i, j, r_val) in terms
                for d in D
            ) <= lambdaVar[a[0], a[1], time] * c[a]
        )

"""
# (c) Load constraints

for time in T:
    ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
    for a in ANotInQ:
        m.addConstr(
            gp.quicksum(r.get((i, j, a[0], a[1], time),0.0) * v[d, time] * x[d, time, i, j] for d in D for (i, j) in pairs)
            <= lambdaVar[a[0], a[1], time] * c[a])
"""

print(f"Load constraints: {time_setting.time()-t4} s")

t5 = time_setting.time()

# (d) Budget constraints
m.addConstrs(gp.quicksum(y[d, time, i, j] for d in D for (i,j) in pairs) <= kappla[time] for time in T_star)
m.addConstrs(y[d, time, i, j] >= x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)
m.addConstrs(y[d, time, i, j] >= -x[d, time, i, j] + x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)

print(f"Budget constraints: {time_setting.time()-t5} s")

t6 = time_setting.time()

"""
# Chat - Add fake constraints to enforce no direct segment path
for d in D:
    for tt in T:
        m.addConstr(x[d, tt, s[d], t[d]] == 0)
        
# Claude - Add fake node-visit count constraints: each intermediate node used at most once per demand/time
for d in D:
    for tt in T:
        sd, td = s[d], t[d]
        for i in V:
            if i != sd and i != td:
                
                # in-degree <= 1
                m.addConstr(gp.quicksum(x[d,tt,j,i] for j in V if j!=i) <= 1)
                
                # out-degree <= 1
                m.addConstr(gp.quicksum(x[d,tt,i,j] for j in V if j!=i) <= 1)
        
        # Source: nothing flows INTO source
        m.addConstr(gp.quicksum(x[d,tt,j,sd] for j in V if j!=sd) == 0)
        
        # Target: nothing flows OUT OF target
        m.addConstr(gp.quicksum(x[d,tt,td,j] for j in V if j!=td) == 0)
"""

# Objective
m.setObjective(0, GRB.MINIMIZE)
m.optimize()

print(f"Objective and optimization step: {time_setting.time()-t6} s")
print(f"Total time for the instance {idx} : {time_setting.time()-t0} s")
print(f"Total time for the instance {idx} : {(time_setting.time()-t0)/60} min")

print("Check of the x :")
for d in D:
    for time in T:
        seg =[]
        for (i,j) in pairs:
            if x[d, time, i, j].X > 0.5:
                seg.append((i,j))
        if seg:
            print(f"d={d} (s={s[d]}, t={t[d]}), time={time}: {seg}")

if m.Status == GRB.OPTIMAL:
    print("Feasible solution")
    srpaths = output_creation(x, D, T, pairs, s, t)
    output = {"srpaths": srpaths}
    # creates the file if it doesn't exist or overwrites
    with open(outputPath, "w") as f:
        # indent=2 to make 2 indentations to be readable
        json.dump(output, f, indent=2)
    print(f" Output of size ({len(srpaths)})")
else:
    print(f"No solution : {m.Status}")

---------------------------------------------------------------------------------------------------------------------------
instance 01
---------------------------------------------------------------------------------------------------------------------------
D : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
s: {0: 7, 1: 15, 2: 2, 3: 16, 4: 9, 5: 8, 6: 18, 7: 17, 8: 0, 9: 18, 10: 9, 11: 15, 12: 14, 13: 7, 14: 19, 15: 4, 16: 14, 17: 5, 18: 10, 19: 6, 20: 15, 21: 5, 22: 16, 23: 12, 24: 19, 25: 16, 26: 15, 27: 17, 28: 11, 29: 6, 30: 15, 31: 11, 32: 19, 33: 8, 34: 9, 35: 1, 36: 16, 37: 7, 38: 7, 39: 3}
t: {0: 8, 1: 0, 2: 3, 3: 12, 4: 6, 5: 17, 6: 14, 7: 0, 8: 18, 9: 3, 10: 14, 11: 8, 12: 15, 13: 14, 14: 17, 15: 11, 16: 16, 17: 16, 18: 18, 19: 0, 20: 9, 21: 10, 22: 3, 23: 9, 24: 16, 25: 8, 26: 19, 27: 15, 28: 15, 29: 9, 30: 4, 31: 19, 32: 8, 33: 15, 34: 16, 35: 16, 36: 11, 37: 18, 38: 2,

Iteration all the instances in the setA

In [ ]:
datasetPath = "../setA"
for idx in range(1,21):
    t0 = time_setting.time()
    if idx < 10:
        instance = f"0{idx}"
    else:
        instance = f"{idx}"
    outputPath = f"../outputSetA//setA-{instance}-srpaths.json"
    print("---------------------------------------------------------------------------------------------------------------------------")
    print(f"instance {instance}")
    print("---------------------------------------------------------------------------------------------------------------------------")
    with open(f"{datasetPath}/setA-{instance}-net.json", "r") as netFile:
        net = json.load(netFile)

    with open(f"{datasetPath}/setA-{instance}-scenario.json", "r") as scenarioFile:
        scenario = json.load(scenarioFile)

    with open(f"{datasetPath}/setA-{instance}-tm.json", "r") as tmFile:
        tm = json.load(tmFile)

    # Network file

    V = [node["id"] for node in net["nodes"]]
    A = [(link["from"], link["to"]) for link in net["links"]]

    # Dict from arc (i, j) to capacity of arc (i, j)
    c = {(link["from"], link["to"]): link["capacity"] for link in net["links"]}

    # Dict from arc (i, j) to weight/metric of arc (i, j)
    weight = {(link["from"], link["to"]): link["metric"] for link in net["links"]}

    # Dict from link id to arc (i, j)
    link_to_A = {link["id"]: (link["from"], link["to"]) for link in net["links"]}

    # Dict from arc (i, j) to link id
    A_to_link = {(link["from"], link["to"]): link["id"] for link in net["links"]}


    # Scenario file

    maxSeg = scenario["max_segments"]

    # Dict from time t to budget value at time t
    kappla = {i["t"]: i["value"] for i in scenario["budget"]}

    # Dict from time t to list of link id (that needs an intervation at time t)
    q = {i["t"]: i["links"] for i in scenario["interventions"]}

    # Intervation expressed in a arc way instead of the id_link
    # Dict from time t to list of arcs (i, j) (that needs an intervation at time t)
    q_arc = {t: [link_to_A[link_id] for link_id in links] for t, links in q.items()}

    # tm file

    num_time_slots = tm["num_time_slots"]
    T = list(range(num_time_slots))
    T_star = T[1:]
    D = list(range(len(tm["demands"])))  # demands indexed 0..n-1
    print(f"D : {D}")

    # Dict from demand d to source node s(d)
    s = {d: tm["demands"][d]["s"] for d in D}
    print(f"s: {s}")

    # Dict from demand d to target node t(d)
    t = {d: tm["demands"][d]["t"] for d in D}
    print(f"t: {t}")

    # Dict from (d, t) to traffic volume of demand d at time time
    v = {(d, time): tm["demands"][d]["v"][time] for d in D for time in T}

    # Dict from (d, t) to traffic volume of demand d at time time
    v = {(d, time): tm["demands"][d]["v"][time] for d in D for time in T}
    
    # Offline computation
    
    pairs = [(i, j) for i in V for j in V if i != j]

    r = computeSplitCoef(pairs, A, T, V, q_arc, net)
    
    print(f"Offline computation: {time_setting.time()-t0} s")
    
    t1 = time_setting.time()
    
    m = gp.Model()

    m.Params.OutputFlag = 0 
    
    # Variable definition

    # Binary variable: 1 if the path for demand d uses segment (i,j) at time t
    x = m.addVars(D, T, V, V, vtype=GRB.BINARY, name="x")

    # Load variable of arc a to time t
    lambdaVar = m.addVars(A, T, lb=0.0, name="lambda")

    # Buffer variable for the absolute value
    y = m.addVars(D, T_star, V, V, lb=0.0, name="y")

    print(f"Variable definition: {time_setting.time()-t1} s")
    
    t2 = time_setting.time()
    
    # Constraint definition

    # (a) Flow conservation constraints
    for d in D:
        sd = s[d]
        td = t[d]
        for time in T:
            for i in V:
                buffer = 0
                if i == sd:
                    buffer = 1
                elif i == td:
                    buffer = -1
                m.addConstr((gp.quicksum(x[d, time, i, j] for j in V if i != j))-gp.quicksum(x[d, time, j, i] for j in V if i != j) == buffer)
    print(f"Flow conservation constraints: {time_setting.time()-t2} s")
    
    t3 = time_setting.time()
    
    # (b) Number of segments
    m.addConstrs(gp.quicksum(x[d, time, i, j] for (i, j) in pairs) <= maxSeg for d in D for time in T)

    print(f"Number of segments constraints: {time_setting.time()-t3} s")
    
    t4 = time_setting.time()
    
    # (c) Load constraints - speeded up - To verify
    
    nonzero_r = {}
    for (i, j, a0, a1, time), val in r.items():
        if val > 1e-9:
            key = ((a0, a1), time)
            if key not in nonzero_r:
                nonzero_r[key] = []
            nonzero_r[key].append((i, j, val))
    
    for time in T:
        ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
        for a in ANotInQ:
            terms = nonzero_r.get((a, time), [])
            if not terms:
                continue
            m.addConstr(
                gp.quicksum(
                    r_val * v[d, time] * x[d, time, i, j]
                    for (i, j, r_val) in terms
                    for d in D
                ) <= lambdaVar[a[0], a[1], time] * c[a]
            )
    
    """
    # (c) Load constraints
    
    for time in T:
        ANotInQ = [(i, j) for (i, j) in A if (i, j) not in q_arc.get(time, [])]
        for a in ANotInQ:
            m.addConstr(
                gp.quicksum(r.get((i, j, a[0], a[1], time),0.0) * v[d, time] * x[d, time, i, j] for d in D for (i, j) in pairs)
                <= lambdaVar[a[0], a[1], time] * c[a])
    """

    print(f"Load constraints: {time_setting.time()-t4} s")
    
    t5 = time_setting.time()

    # (d) Budget constraints
    m.addConstrs(gp.quicksum(y[d, time, i, j] for d in D for (i,j) in pairs) <= kappla[time] for time in T_star)
    m.addConstrs(y[d, time, i, j] >= x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)
    m.addConstrs(y[d, time, i, j] >= -x[d, time, i, j]- x[d, time-1, i, j] for d in D for (i,j) in pairs for time in T_star)

    print(f"Budget constraints: {time_setting.time()-t5} s")
    
    t6 = time_setting.time()
    
    """
    # Chat - Add fake constraints to enforce no direct segment path
    for d in D:
        for tt in T:
            m.addConstr(x[d, tt, s[d], t[d]] == 0)
            
    # Claude - Add fake node-visit count constraints: each intermediate node used at most once per demand/time
    for d in D:
        for tt in T:
            sd, td = s[d], t[d]
            for i in V:
                if i != sd and i != td:
                    
                    # in-degree <= 1
                    m.addConstr(gp.quicksum(x[d,tt,j,i] for j in V if j!=i) <= 1)
                    
                    # out-degree <= 1
                    m.addConstr(gp.quicksum(x[d,tt,i,j] for j in V if j!=i) <= 1)
            
            # Source: nothing flows INTO source
            m.addConstr(gp.quicksum(x[d,tt,j,sd] for j in V if j!=sd) == 0)
            
            # Target: nothing flows OUT OF target
            m.addConstr(gp.quicksum(x[d,tt,td,j] for j in V if j!=td) == 0)
    """
    
    # Objective
    m.setObjective(0, GRB.MINIMIZE)
    m.optimize()

    print(f"Objective and optimization step: {time_setting.time()-t6} s")
    print(f"Total time for the instance {idx} : {time_setting.time()-t0} s")
    print(f"Total time for the instance {idx} : {(time_setting.time()-t0)/60} min")
    
    print("Check of the x :")
    for d in D:
        for time in T:
            seg =[]
            for (i,j) in pairs:
                if x[d, time, i, j].X > 0.5:
                    seg.append((i,j))
            if seg:
                print(f"d={d} (s={s[d]}, t={t[d]}), time={time}: {seg}")
        
    if m.Status == GRB.OPTIMAL:
        print("Feasible solution")
        print("Feasible solution")
        srpaths = output_creation(x, D, T, pairs, s, t)
        output = {"srpaths": srpaths}
        # creates the file if it doesn't exist or overwrites
        with open(outputPath, "w") as f:
            # indent=2 to make 2 indentations to be readable
            json.dump(output, f, indent=2)
        print(f" Output of size ({len(srpaths)})")
    else:
        print(f"No solution : {m.Status}")

---------------------------------------------------------------------------------------------------------------------------
instance 01
---------------------------------------------------------------------------------------------------------------------------
D : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
s: {0: 7, 1: 15, 2: 2, 3: 16, 4: 9, 5: 8, 6: 18, 7: 17, 8: 0, 9: 18, 10: 9, 11: 15, 12: 14, 13: 7, 14: 19, 15: 4, 16: 14, 17: 5, 18: 10, 19: 6, 20: 15, 21: 5, 22: 16, 23: 12, 24: 19, 25: 16, 26: 15, 27: 17, 28: 11, 29: 6, 30: 15, 31: 11, 32: 19, 33: 8, 34: 9, 35: 1, 36: 16, 37: 7, 38: 7, 39: 3}
t: {0: 8, 1: 0, 2: 3, 3: 12, 4: 6, 5: 17, 6: 14, 7: 0, 8: 18, 9: 3, 10: 14, 11: 8, 12: 15, 13: 14, 14: 17, 15: 11, 16: 16, 17: 16, 18: 18, 19: 0, 20: 9, 21: 10, 22: 3, 23: 9, 24: 16, 25: 8, 26: 19, 27: 15, 28: 15, 29: 9, 30: 4, 31: 19, 32: 8, 33: 15, 34: 16, 35: 16, 36: 11, 37: 18, 38: 2,

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001BEA019DBE0>>
Traceback (most recent call last):
  File "c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel\ipkernel.py", line 796, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
  File "c:\Users\antoi\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1479, in enumerate
    def enumerate():
KeyboardInterrupt: 


Variable definition: 0.8960103988647461 s
Flow conservation constraints: 0.22852611541748047 s
Number of segments constraints: 0.09818053245544434 s


Algorithms and heuristics

In [ ]:
# Objective
m.update()
zlist = []
for a in A:
    for time in T:
        z = m.addVar(lb=0.0)
        tempConstrs = m.addConstrs(lambdaVar[a[0], a[1], time] <= z for a in A for time in T)
        m.setObjective(z, GRB.MINIMIZE)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            break
        zlist.append(z.X)
        m.addConstrs(lambdaVar[a[0], a[1], time] <= z.X for a in A for time in T)
        m.remove(list(tempConstrs.values()))
        m.remove(z)
        if len(zlist) > 1 and abs(zlist[-1] - zlist[-2]) < 1e-6:
            break

Results and performance analysis